In [ ]:
import cv2
import os
import json
import joblib
import sys
import pandas as pd
import numpy as np
import mediapipe as mp
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

# =====================================================================
# 🛠️ 1. KONFIGURACJA TESTU (WYPEŁNIJ PRZED URUCHOMIENIEM)
# =====================================================================
SESSION_NAME = "tylko_zla1_laptop_wyzej"  # Nazwa folderu w data/raw_frames/
PROFILE_NAME = "laptop_wyzej"          # Jakiego profilu kalibracji używamy?
NOTATKI = "laptop_wyzej, tylko zla pochylona do przodu postawa"

# Zdefiniuj przedziały jako NUMERY KLATEK (z Twoich notatek)
# Format: (klatka_start, klatka_koniec, postawa_0_lub_1)
GROUND_TRUTH_FRAMES = [
    (0, 10544, 1)
]
# =====================================================================

# Ścieżki
BASE_DIR = os.path.abspath("../../")
if BASE_DIR not in sys.path:
    sys.path.append(BASE_DIR)
RAW_FRAMES_DIR = os.path.join(BASE_DIR, "data", "raw_frames", SESSION_NAME)
REPORTS_DIR = os.path.join(BASE_DIR, "data", "reports", SESSION_NAME)
SUMMARY_CSV = os.path.join(BASE_DIR, "data", "reports", "podsumowanie_wszystkich_testow.csv")

MODEL_PATH = os.path.join(BASE_DIR, "models", "posture_model.pkl")
PROFILE_PATH = os.path.join(BASE_DIR, "data", "profiles", f"{PROFILE_NAME}.json")

os.makedirs(REPORTS_DIR, exist_ok=True)

if not os.path.exists(RAW_FRAMES_DIR):
    print(f"❌ BŁĄD: Nie znaleziono folderu z klatkami: {RAW_FRAMES_DIR}")
elif not os.path.exists(MODEL_PATH):
    print(f"❌ BŁĄD: Nie znaleziono modelu w {MODEL_PATH}")
elif not os.path.exists(PROFILE_PATH):
    print(f"❌ BŁĄD: Nie znaleziono profilu '{PROFILE_NAME}' w {PROFILE_PATH}")
else:
    print(f"🚀 Rozpoczynam analizę sesji: {SESSION_NAME}")
    
    # 2. Ładowanie modelu
    from src.core.normalizer import normalize_features
    model = joblib.load(MODEL_PATH)
    
    with open(PROFILE_PATH, 'r', encoding='utf-8') as f:
        calibration = json.load(f)['calibration']
    
    mp_pose = mp.solutions.pose
    LANDMARKS_TO_USE = {"nose": 0, "left_eye": 2, "right_eye": 5, "left_ear": 7, "right_ear": 8, "left_shoulder": 11, "right_shoulder": 12}
    
    def extract_features(landmarks):
        features = []
        for name, idx in LANDMARKS_TO_USE.items():
            features.extend([landmarks[idx].x, landmarks[idx].y, landmarks[idx].z])
        return features

    # NOWA Funkcja działająca na klatkach
    def get_ground_truth(current_frame, intervals):
        for start, end, label in intervals:
            if start <= current_frame <= end:
                return label
        return 0 # Jeśli klatka ucieknie poza przedział, zakładamy dobrą postawę

    # 3. Pętla przetwarzająca klatki
    image_files = sorted([f for f in os.listdir(RAW_FRAMES_DIR) if f.endswith('.jpg') or f.endswith('.png')])
    total_frames = len(image_files)
    print(f"📸 Znaleziono {total_frames} klatek do analizy. Uruchamiam AI (to może potrwać kilka minut)...")
    
    results_list = []
    
    with mp_pose.Pose(static_image_mode=True, min_detection_confidence=0.5, model_complexity=1) as pose:
        for idx, img_name in enumerate(image_files):
            # Prosty pasek postępu w konsoli, żebyś wiedziała, że skrypt nie "zamarzł"
            if idx % 1000 == 0 and idx > 0:
                print(f"Przeanalizowano: {idx} / {total_frames} klatek...")

            img_path = os.path.join(RAW_FRAMES_DIR, img_name)
            frame = cv2.imread(img_path)
            if frame is None: continue
            frame = cv2.flip(frame, 1)
            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = pose.process(rgb_frame)
            
            true_label = get_ground_truth(idx, GROUND_TRUTH_FRAMES)
            pred_label = -1
            confidence = 0.0
            
            if results.pose_landmarks:
                raw_feats = extract_features(results.pose_landmarks.landmark)
                norm_feats = normalize_features(raw_feats, calibration)
                X = np.array(norm_feats).reshape(1, -1)
                
                pred_label = int(model.predict(X)[0])
                confidence = float(max(model.predict_proba(X)[0]))
                
            results_list.append({
                'frame_num': idx,
                'ground_truth': true_label,
                'prediction': pred_label,
                'confidence': round(confidence, 2)
            })

   # 4. Zapis surowych wyników i WYGŁADZANIE (MIMIKRA SERWERA)
    df = pd.DataFrame(results_list)
    df = df[df['prediction'] != -1] # Wyrzucamy klatki bez postaci
    
    # --- DODAJEMY WYGŁADZANIE (Dokładnie jak SMOOTH_N = 20 w server.py) ---
    SMOOTH_N = 20
    # Obliczamy średnią z ostatnich 20 klatek. Jeśli >= 70% to zła postawa (1), to dajemy 1
    df['smoothed_prediction'] = df['prediction'].rolling(window=SMOOTH_N, min_periods=1).mean()
    df['prediction'] = df['smoothed_prediction'].apply(lambda x: 1 if x >= 0.7 else 0)
    # ----------------------------------------------------------------------

    df.to_csv(os.path.join(REPORTS_DIR, "wyniki_detekcji.csv"), index=False)
    
    acc = accuracy_score(df['ground_truth'], df['prediction'])
    print(f"\n🎯 Gotowe! Dokładność wygładzonego modelu: {acc*100:.2f}%")
    # 5. Aktualizacja ZBIORCZEJ TABELI
    summary_data = pd.DataFrame([{
        "Nazwa Sesji": SESSION_NAME,
        "Użyty Profil": PROFILE_NAME,
        "Dokładność (%)": round(acc*100, 2),
        "Zbadanych Klatek": len(df),
        "Notatki": NOTATKI
    }])
    summary_data.to_csv(SUMMARY_CSV, mode='a', header=not os.path.exists(SUMMARY_CSV), index=False)
    
    # ==========================================
    # 6. Generowanie Macierzy Konfuzji (Obraz)
    # ==========================================
    cm = confusion_matrix(df['ground_truth'], df['prediction'], labels=[0, 1])
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Faktycznie: Dobra (0)', 'Faktycznie: Zła (1)'], 
                yticklabels=['AI: Dobra (0)', 'AI: Zła (1)'])
    plt.title(f'Macierz Konfuzji\nSesja: {SESSION_NAME}', pad=15)
    plt.tight_layout()
    plt.savefig(os.path.join(REPORTS_DIR, "macierz_konfuzji.png"), dpi=300, bbox_inches='tight')
    plt.close()

    # ==========================================
    # 7. Generowanie Wykresu - OŚ KLATEK
    # ==========================================
    fig, ax = plt.subplots(figsize=(14, 4))
    
    # Tło prawdziwej postawy (używamy numerów klatek!)
    ax.fill_between(df['frame_num'], -0.1, 1.1, where=df['ground_truth'] == 0, color='green', alpha=0.15, label='Prawda: Dobra')
    ax.fill_between(df['frame_num'], -0.1, 1.1, where=df['ground_truth'] == 1, color='red', alpha=0.15, label='Prawda: Zła')
    
    # Linia AI
    ax.step(df['frame_num'], df['prediction'], where='post', color='#2c3e50', linewidth=1.5, label='Decyzja AI')
    
    ax.set_yticks([0, 1])
    ax.set_yticklabels(['DOBRA (0)', 'ZŁA (1)'], fontweight='bold')
    
    plt.title(f'Analiza stabilności modelu (wg klatek)\nSesja: {SESSION_NAME} | Profil: {PROFILE_NAME}')
    plt.xlabel('Numer Klatki')
    plt.legend(loc="lower right", framealpha=0.9)
    plt.grid(axis='x', linestyle=':', alpha=0.6)
    plt.tight_layout()
    plt.savefig(os.path.join(REPORTS_DIR, "wykres_czasowy.png"), dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"\n✨ Wszystko zapisano bezpiecznie w folderze: data/reports/{SESSION_NAME}/")

🚀 Rozpoczynam analizę sesji: tylko_zla1_laptop_wyzej
📸 Znaleziono 10545 klatek do analizy. Uruchamiam AI (to może potrwać kilka minut)...
